In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [2]:
# 다양성 
# 관련성
# 균형

In [3]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

In [4]:
examples = [
    {"input" : "이 제품 정말 최고입니다!", "output" : "긍정"},
    {"input" : "품질이 너무 안좋아요", "output" : "부정"},
    {"input" : "보통이에요, 무난합니다.", "output" : "중립"},
    {"input" : "디자인은 좋은데 성능이 아쉬워요", "output" : "혼합"}
]

In [5]:
example_template = PromptTemplate(
    input_variables = ["input", "output"],
    template = "리뷰: {input}\n감정: {output}"
)

fewshot_prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_template,
    prefix = '다음 예시를 참고해서 리뷰의 감정을 분류해주세요.\n',
    suffix = '\n리뷰: {input}\n감정:',
    input_variables = ['input'],
    example_separator = '\n\n'
)

In [6]:
formatted = fewshot_prompt.format(input = '가격은 비싸지만 품질이 뛰어나요')
print(formatted)

다음 예시를 참고해서 리뷰의 감정을 분류해주세요.


리뷰: 이 제품 정말 최고입니다!
감정: 긍정

리뷰: 품질이 너무 안좋아요
감정: 부정

리뷰: 보통이에요, 무난합니다.
감정: 중립

리뷰: 디자인은 좋은데 성능이 아쉬워요
감정: 혼합


리뷰: 가격은 비싸지만 품질이 뛰어나요
감정:


In [7]:
example_prompt_chat = ChatPromptTemplate.from_messages([
    ('human', '리뷰 : {input}'),
    ('ai', '감정: {output}')
])

fewshot_chat_prompt = FewShotChatMessagePromptTemplate(
    examples = examples,
    example_prompt = example_prompt_chat
)

In [8]:
final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 감정 분석 전문가입니다, 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류해주세요'),
    fewshot_chat_prompt,
    ('human', '리뷰 : {input}')
])

In [9]:
chain = final_prompt | llm 
result = chain.invoke({'input' : '가격은 비싸지만 품질이 뛰어나요'})
print(result.content)

감정: 혼합


In [10]:
large_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
    {"input": "고객센터 응대가 너무 불친절합니다.", "output": "부정"},
    {"input": "택배가 분실되었어요.", "output": "부정"},
    {"input": "보통이에요. 특별한 점은 없습니다.", "output": "중립"},
    {"input": "사진과 동일한 제품입니다.", "output": "중립"},
    {"input": "디자인은 예쁜데 내구성이 약해요.", "output": "혼합"},
    {"input": "기능은 많은데 사용법이 복잡합니다.", "output": "혼합"},
]


In [11]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

In [12]:
from langchain_core.vectorstores import InMemoryVectorStore

In [13]:
example_selector = SemanticSimilarityExampleSelector.from_examples(
    large_examples,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    InMemoryVectorStore,
    k=3
)

In [14]:
queries = [
    '배송이 너무 느려요',
    '값은 좀 나가지만 성능은 훌륭해요',
    '무난한 제품이에요'
]

for q in queries:
    selected = example_selector.select_examples({'input' : q})
    print(f"query : {q}")
    for s in selected:
        print(f" selected : {s['output']}, {s['input']}")

query : 배송이 너무 느려요
 selected : 긍정, 배송이 빠르고 포장이 꼼꼼해요.
 selected : 부정, 고객센터 응대가 너무 불친절합니다.
 selected : 부정, 택배가 분실되었어요.
query : 값은 좀 나가지만 성능은 훌륭해요
 selected : 긍정, 가격 대비 훌륭합니다.
 selected : 혼합, 기능은 많은데 사용법이 복잡합니다.
 selected : 혼합, 디자인은 예쁜데 내구성이 약해요.
query : 무난한 제품이에요
 selected : 중립, 사진과 동일한 제품입니다.
 selected : 부정, 품질이 최악입니다. 환불 요청했어요.
 selected : 긍정, 가격 대비 훌륭합니다.


In [15]:
# 동적으로 선택 -> FewShotChatMessagePromptTemplate -> llm 답변

In [16]:
dynamic_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt_chat
)

dynamic_final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 감정 분석 전문가입니다. 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류하세요'),
    dynamic_few_shot_prompt,
    ('human', '리뷰 : {input}')
])

dynamic_chain = dynamic_final_prompt | llm


In [17]:
test_inputs  = ['택배 상자가 찌그러져왔어요', '화면은 선명한데 스피커 소리가 작아요']
for t in test_inputs:
    result = dynamic_chain.invoke({'input' : t})
    print(f'{t} : {result.content}')

택배 상자가 찌그러져왔어요 : 감정: 부정
화면은 선명한데 스피커 소리가 작아요 : 감정: 혼합


In [18]:
eval_dataset = [
    {"text": "완벽한 제품입니다! 강력 추천합니다.", "expected": "긍정"},
    {"text": "두 번 다시 구매하지 않겠습니다.", "expected": "부정"},
    {"text": "평범합니다. 특별히 좋지도 나쁘지도 않아요.", "expected": "중립"},
    {"text": "기능은 좋은데 AS가 아쉬워요.", "expected": "혼합"},
    {"text": "가격도 착하고 품질도 좋아요.", "expected": "긍정"},
    {"text": "사진과 너무 달라서 실망했습니다.", "expected": "부정"},
    {"text": "그냥 쓸만 합니다.", "expected": "중립"},
    {"text": "배송은 빠른데 제품이 기대 이하에요.", "expected": "혼합"},
]


In [19]:
def evaluate_zero_shot(dataset):
    results = []
    for item in dataset:
        response = llm.invoke([
            SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요"),
            HumanMessage(content = item['text'])
        ]).content
        
        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })
        
    return results

In [20]:
def evaluate_few_shot(dataset, examples):
    results = []
    base_messages = [SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요")]
    for ex in examples:
        base_messages.append(HumanMessage(content = ex['input']))
        base_messages.append(AIMessage(content = ex['output']))
        
    for item in dataset:
        messages = base_messages + [HumanMessage(content = item['text'])]
        response = llm.invoke(messages).content
                
        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })
        
    return results

In [21]:
few_shot_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
]

zero_results = evaluate_zero_shot(eval_dataset)

In [22]:
fewshot_results = evaluate_few_shot(eval_dataset, few_shot_examples)

In [23]:
zero_results

[{'text': '완벽한 제품입니다! 강력 추천합니다.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '두 번 다시 구매하지 않겠습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '평범합니다. 특별히 좋지도 나쁘지도 않아요.',
  'expected': '중립',
  'predicted': '중립',
  'correct': True},
 {'text': '기능은 좋은데 AS가 아쉬워요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True},
 {'text': '가격도 착하고 품질도 좋아요.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '사진과 너무 달라서 실망했습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '그냥 쓸만 합니다.', 'expected': '중립', 'predicted': '중립', 'correct': True},
 {'text': '배송은 빠른데 제품이 기대 이하에요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True}]

In [24]:
fewshot_results

[{'text': '완벽한 제품입니다! 강력 추천합니다.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '두 번 다시 구매하지 않겠습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '평범합니다. 특별히 좋지도 나쁘지도 않아요.',
  'expected': '중립',
  'predicted': '중립',
  'correct': True},
 {'text': '기능은 좋은데 AS가 아쉬워요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True},
 {'text': '가격도 착하고 품질도 좋아요.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '사진과 너무 달라서 실망했습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '그냥 쓸만 합니다.', 'expected': '중립', 'predicted': '중립', 'correct': True},
 {'text': '배송은 빠른데 제품이 기대 이하에요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True}]

In [25]:
import pandas as pd

In [26]:
pd.DataFrame(fewshot_results) # .to_csv('a.csv')

,text,expected,predicted,correct
0,완벽한 제품입니다! 강력 추천합니다.,긍정,긍정,True
1,두 번 다시 구매하지 않겠습니다.,부정,부정,True
2,평범합니다. 특별히 좋지도 나쁘지도 않아요.,중립,중립,True
3,기능은 좋은데 AS가 아쉬워요.,혼합,혼합,True
4,가격도 착하고 품질도 좋아요.,긍정,긍정,True
5,사진과 너무 달라서 실망했습니다.,부정,부정,True
6,그냥 쓸만 합니다.,중립,중립,True
7,배송은 빠른데 제품이 기대 이하에요.,혼합,혼합,True


In [27]:
# classification related metric

In [28]:
# 1.0      0.9    : root((1-0.9)**2) 

# 0.8,     0.5    :
# 0.9      0.1
# 0.0      0.2

In [29]:
# regression related metric : rmse (root mean squared error) , mae mse...

In [30]:
# 번역, 요약, 문구 생성 : BLUE, LOUGE
# 나는 학교에 갑니다 => I go to school  : I go school  0.75

In [ ]:
말이 25마리, 한번에 5마리만 경주를 할 수 있다
가장 빠른 3마리 말을 찾는게 목표
몇번의 경주를 해야될까요?

In [32]:
result = llm.invoke('말이 25마리 있습니다. 한 번에 5마리만 경주를 할 수 있습니다. 가장 빠른 3마리 말을 찾으려면 몇 번의 경주를 해야할까요?')

In [33]:
print(result.content)

25마리의 말 중에서 가장 빠른 3마리를 찾기 위해 가장 효율적으로 경주를 진행하는 방법을 살펴보겠습니다.

1. **1단계**: 25마리를 5마리씩 5개의 그룹으로 나누고 각각의 그룹에서 1차 경주를 합니다. 이 단계에서 5번의 경주가 필요합니다.
   
2. **2단계**: 각 그룹의 1위, 2위, 3위 말을 뽑아서 다음 경주를 위해 총 15마리(그룹에서 1위 5마리)로 경주를 진행합니다. 이 단계에서 1번의 경주가 필요합니다.

3. **3단계**: 2단계에서 경주 결과를 바탕으로 최종 빠른 말을 결정합니다. 이제 놓칠 수 있는 말을 고려해야 합니다. 
   - 2위와 3위의 말을 찾기 위해 2단계 경주에서 1위의 말 다음에 속한 말들, 즉 최종 경주에서 1위의 그룹에서 2위, 3위 말과 2위의 그룹에서 1위의 말과 3위의 그룹에서 1위의 말을 경주에 포함시킵니다.
   - 반드시 포함해야 할 말은 1위의 그룹에서 2위와 3위, 2위의 그룹에서 1위 말, 3위의 그룹에서 1위 말, 총 5마리입니다.

이렇게 하여 추가로 1번의 경주를 진행하면 됩니다.

결과적으로:
- 1단계: 5번 경주
- 2단계: 1번 경주
- 3단계: 1번 경주

최종적으로는 5 + 1 + 1 = **7번의 경주**가 필요합니다. 따라서 25마리의 말 중에서 가장 빠른 3마리를 찾기 위해서는 총 7번의 경주가 필요합니다.


In [34]:
result = llm.invoke('말이 6마리가 있고, 가장 빠른 말을 찾고 싶습니다. 한 번에 최대 6마리가 동시에 뛸 수 있습니다. 최소 몇 번의 경주가 필요할까요?')

In [35]:
print(result.content)

가장 빠른 말을 찾기 위해서는 최소한 7번의 경주가 필요합니다. 아래는 그 이유를 설명합니다.

1. **첫 번째 경주 (1~6번):** 6마리를 동시에 경주시킨다. 이 경주를 통해 각 말의 순위를 얻을 수 있습니다. 
   - 예를 들어, 결과가 A(1등), B(2등), C(3등), D(4등), E(5등), F(6등)라고 가정합니다.

2. **두 번째 경주 (A, B, C):** 첫 번째 경주에서 1, 2, 3등을 차지한 A, B, C 세 마리만을 대상으로 경주를 실시합니다.
   - 이 경주의 결과에 따라 가장 빠른 말을 결정할 수 있습니다.

이 과정을 통해 모든 가능성을 고려하면, 7번의 경주가 필요하는 것입니다.


In [36]:
llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요?')

AIMessage(content='이 문제는 "몬티 홀 문제"로 알려진 유명한 확률 퍼즐과 관련이 있습니다. 문제를 좀 더 자세히 설명하자면:\n\n1. 세 개의 문(A, B, C) 중 하나에 상금(황금)이 숨겨져 있고 나머지 두 문에는 야채가 있습니다.\n2. 당신이 1번 문(A)을 선택합니다.\n3. 사회자는 당신이 선택하지 않은 문 두 개 중 하나를 열지 않고, 다른 문으로 바꾸도록 요청합니다. (2번 문(B)을 언급하며, 그 문을 열지 않는다고 가정합니다.)\n\n여기서 중요한 점은 사회자가 황금이 없는 문을 열지 않기 때문에 당신의 선택에 따라 확률이 바뀌게 됩니다.\n\n- 처음 선택했을 때, 황금이 1번 문에 있을 확률은 1/3, 2번 문에 있을 확률은 1/3, 3번 문에 있을 확률은 1/3입니다.\n- 당신이 1번 문을 고른 후, 사회자는 황금이 없는 문(B 또는 C)을 열지 않기 때문에 선택을 바꿀 경우 새로운 확률이 생깁니다. \n\n결과적으로:\n- 만약 당신이 선택을 바꾸지 않으면, 여전히 황금이 있을 확률은 1/3입니다.\n- 만약 당신이 선택을 바꾼다면, 남은 문 중 하나에 황금이 있을 확률은 2/3입니다.\n\n따라서, 선택을 바꾸는 것이 유리합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 340, 'prompt_tokens': 74, 'total_tokens': 414, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name'

In [37]:
print('이 상황은 "몬티 홀 문제"라고 불리는 유명한 확률 문제와 유사합니다. 문제를 요약하자면, 3개의 문 중 하나에는 황금이 있고 나머지 2개 문에는 야채가 있습니다. 당신이 1번 문을 선택했을 때, 사회자는 다른 문(예: 2번 문)을 언급하며 이를 바꿀 것인지 물어봅니다.\n\n이 문제에서 중간에 문을 열지 않고 바꾸는 것이 유리한지 분석해보겠습니다.\n\n1. 당신이 처음에 1번 문을 선택했을 때, 그 문에 황금이 있을 확률은 1/3입니다. 나머지 두 문(2번, 3번)에 황금이 있을 확률의 합은 2/3입니다.\n\n2. 사회자가 2번 문으로 바꾸는 방법은 황금이 2번 문에 없을 때만 가능합니다. 따라서 만약 1번 문에 황금이 없다면, 사회자는 항상 3번 문을 열어줍니다. \n\n결론적으로,\n- 바꾸지 않으면 황금이 있을 확률은 1/3입니다.\n- 바꾸면 황금이 있을 확률은 2/3입니다.\n\n따라서, 문을 바꾸는 것이 훨씬 유리합니다.')

이 상황은 "몬티 홀 문제"라고 불리는 유명한 확률 문제와 유사합니다. 문제를 요약하자면, 3개의 문 중 하나에는 황금이 있고 나머지 2개 문에는 야채가 있습니다. 당신이 1번 문을 선택했을 때, 사회자는 다른 문(예: 2번 문)을 언급하며 이를 바꿀 것인지 물어봅니다.

이 문제에서 중간에 문을 열지 않고 바꾸는 것이 유리한지 분석해보겠습니다.

1. 당신이 처음에 1번 문을 선택했을 때, 그 문에 황금이 있을 확률은 1/3입니다. 나머지 두 문(2번, 3번)에 황금이 있을 확률의 합은 2/3입니다.

2. 사회자가 2번 문으로 바꾸는 방법은 황금이 2번 문에 없을 때만 가능합니다. 따라서 만약 1번 문에 황금이 없다면, 사회자는 항상 3번 문을 열어줍니다. 

결론적으로,
- 바꾸지 않으면 황금이 있을 확률은 1/3입니다.
- 바꾸면 황금이 있을 확률은 2/3입니다.

따라서, 문을 바꾸는 것이 훨씬 유리합니다.


In [38]:
# Sally(여자) 한테 남자 형제가 3명 있어요
# 각 남자형제한테는 여자 형제가 2명 있어요
# Sally 의 여자 형제는 몇명?

In [39]:
print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요?').content)

"LOLLAPALOOZA"라는 단어에서 L은 총 3번 나타납니다.


In [40]:
# print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요?').content)

In [41]:
# CoT (Chain-of-Thought)
# "단계별로 생각해보세요"

In [42]:
result = llm.invoke('말이 6마리가 있고, 가장 빠른 말을 찾고 싶습니다. 한 번에 최대 6마리가 동시에 뛸 수 있습니다. 최소 몇 번의 경주가 필요할까요?')

In [43]:
llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다')

AIMessage(content='이 문제는 유명한 "몬티 홀 문제"에 해당합니다. 문제의 상황을 단계별로 살펴보겠습니다.\n\n1. **초기 선택**: 당신은 3개의 문(A, B, C) 중 하나를 선택합니다. 여기서 황금이 있는 문이 1개, 야채가 있는 문이 2개입니다. 예를 들어, 당신이 1번 문을 선택했다고 가정해 보겠습니다.\n\n2. **가능성 분석**:\n   - 황금이 1번 문에 있을 확률: 1/3\n   - 황금이 2번 문에 있을 확률: 1/3\n   - 황금이 3번 문에 있을 확률: 1/3\n\n3. **사회자의 행동**: 사회자는 선택한 1번 문을 열지 않고, 나머지 문 중에서 야채가 있는 문을 열도록 합니다. 이는 사회자가 항상 황금이 있는 문은 열지 않는다는 전제하에 이루어집니다. 만약 당신이 선택한 문에 황금이 없었다면, 사회자는 반드시 야채가 있는 문을 열어야 합니다.\n\n4. **선택 후의 확률**:\n   - 당신이 1번 문에 선택한 경우, 황금이 있는 문이 1번에 있을 확률은 여전히 1/3입니다. 야채가 있는 경우는 2번이나 3번 중에 분포되어 있습니다.\n   - 사회자가 2번 문을 열었을 경우, 3번 문에 황금이 있을 확률은 2/3로 높아집니다.\n\n5. **전략**:\n   - 처음 선택한 1번 문을 그대로 유지할 경우, 황금이 있을 확률: 1/3\n   - 다른 문(3번 문)으로 바꿀 경우, 황금이 있을 확률: 2/3\n\n결론적으로, 사회자가 문을 변경할 수 있는 경우, 바꾸는 것이 유리합니다. 문을 바꾸면 황금이 있는 문을 선택할 확률이 증가하기 때문입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 455, 'prompt_tokens': 83, 'total_tokens': 538, 'completion_tokens_details': {'accepted_prediction_tokens': 0

In [44]:
print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요? 단계별로 생각해보세요').content)

'LOLLAPALOOZA'라는 단어에서 L의 개수를 세기 위해 단계별로 살펴보겠습니다.

1. 단어를 하나씩 확인합니다: L, O, L, L, A, P, A, L, O, O, Z, A.
2. L이 있는 위치를 체크합니다:
   - 1번째 문자: L
   - 2번째 문자: O
   - 3번째 문자: L
   - 4번째 문자: L
   - 5번째 문자: A
   - 6번째 문자: P
   - 7번째 문자: A
   - 8번째 문자: L
   - 9번째 문자: O
   - 10번째 문자: O
   - 11번째 문자: Z
   - 12번째 문자: A

3. 이제 L이 몇 번 나타나는지 세어봅니다:
   - 1번째: L (1회)
   - 3번째: L (2회)
   - 4번째: L (3회)
   - 8번째: L (4회)

결론적으로, 'LOLLAPALOOZA'라는 단어에서 L은 총 4번 나타납니다.


In [45]:
print(llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다').content)

이 문제는 "몬티 홀 문제"로 알려져 있으며, 확률 이론에 관한 흥미로운 예시입니다. 단계별로 생각해보겠습니다.

### 1단계: 상황 이해하기
- 3개의 문이 있습니다: 1번, 2번, 3번
- 그 중 한 문 뒤에는 황금이 있고 (당첨), 나머지 두 문 뒤에는 야채가 있습니다 (꽝).
- 당신은 1번 문을 선택했습니다.

### 2단계: 선택 후의 확률
- 당신이 처음 선택한 1번 문 뒤에 황금이 있을 확률은 1/3입니다.
- 반면, 나머지 두 문 (2번과 3번) 뒤에 황금이 있을 확률은 합쳐서 2/3입니다.

### 3단계: 사회자의 행동
- 사회자는 당신이 선택하지 않은 문 중 하나를 열어, 그 문 뒤에 야채가 있음을 보여줍니다.
- 예를 들어, 만약 2번 문을 열어 야채를 보여주었다면, 이 경우 3번 문에도 황금이 있을 확률이 2/3로 남아 있게 됩니다.

### 4단계: 선택 변경의 이점
- 만약 처음 선택한 1번 문에 황금이 있을 확률은 여전히 1/3이고, 이제 남은 문 중 하나인 3번 문 뒤에 황금이 있을 확률은 2/3입니다.
- 따라서 1번 문에서 3번 문으로 바꾸는 것이 유리합니다.

### 5단계: 결론
- 문을 바꾸면 황금이 있을 확률이 2/3로 증가하므로, 바꾸는 것이 수학적으로 더 유리합니다.

결론적으로, 문을 바꾸는 것이 더 좋은 전략입니다.


In [46]:
print(llm.invoke("""Sally(여자) 한테 남자 형제가 3명 있어요
각 남자형제한테는 여자 형제가 2명 있어요
Sally 의 여자 형제는 몇명인가요?
단계별로 생각해보세요""").content)

Sally에게 남자 형제가 3명 있다고 하셨습니다. 이제 각각의 남자 형제에게 여자 형제가 2명씩 있다고 합니다.

1. **Sally의 남자 형제 수**: 3명
2. **각 남자 형제에게 있는 여자 형제 수**: 2명

이때 "여자 형제"란 Sally를 포함한 여자 형제를 의미하므로 각 남자 형제에게는 Sally 외에 다른 여자 형제도 있을 수 있습니다.

- 첫번째 남자 형제: Sally + 여자 형제 1명
- 두번째 남자 형제: Sally + 여자 형제 1명
- 세번째 남자 형제: Sally + 여자 형제 1명

이 경우, Sally는 모든 남자 형제의 여자 형제에 포함됩니다. 즉, 당신이 언급한 "여자 형제"는 Sally를 포함한 총 여자 형제입니다.

3. 각 남자 형제는 Sally를 제외한 추가적인 여자 형제가 등이 없다면, Sally 혼자만이 여자 형제가 됩니다.

따라서 Sally의 여자 형제 수는 **0명**입니다.


In [47]:
# CoT
# 단계별로 생각해보세요, 봅시다
# 지문의 조건을 전부 나열 - 각각의 조건들을 검토하면서 추론해줘/풀어줘
# 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘

In [48]:
print(llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다, 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘').content)

이 문제는 유명한 "문제의 문" 또는 "몽티 홀 문제"의 변형입니다. 전통적인 몽티 홀 문제에서는 세 개의 문 중 하나에 상이 있으며, 참가자가 선택한 후 사회자가 무작위로 염소(또는 상금이 없는 문)를 열고, 참가자가 문을 바꿀 것이냐 물었습니다.

지금 주어진 조건에서, 사회자가 2번 문으로 바꾸겠냐고 묻는 상황은 기본적으로 동일한 원칙을 따릅니다. 문을 바꾸는 것이 유리한지를 단계별로 분석해 보겠습니다.

1. **초기 선택**: 참가자가 1번 문을 고릅니다. 이 경우, 다음과 같은 상황이 있습니다.
   - 황금이 1번 문 뒤에 있을 확률 = 1/3
   - 황금이 2번 문 뒤에 있을 확률 = 1/3
   - 황금이 3번 문 뒤에 있을 확률 = 1/3

2. **사회자의 행동**: 사회자는 2번 문을 고르라고 제안했는데, 이는 황금이 없는 문 중 하나를 고르고 열어야 하는 규칙이 지켜진 것으로 가정합니다. 하지만 특정 문제에서 사회자가 이미 선택한 2번 문으로 바꿨다는 것은 그 문이 황금을 가지지 않는 문이라는 전제가 필요합니다. 이 경우, 만약 2번 문이 황금을 가지고 있다면, 사회자가 2번 문을 오픈하지 않는 이유가 설명되지 않습니다.

3. **바꾸기 선택**: 이제 기존의 몽티 홀 문제와 같은 방식으로 바꿔 볼까요?
   - 1번 문에서 황금이 있을 확률은 고정된 채로 1/3입니다.
   - 2번 문으로 바꾸면, 황금이 2번 문에도 있을 확률이 증가하지 않습니다.
   - 반대로, 사회자가 열 수 있는 문이 황금이 없다는 맥락에서, 대부분의 경우 3번 문에 황금이 있을 확률은 2/3으로 증가하는 것입니다.

4. **결론**: 결과적으로 사회자가 해당 문으로 바꾸는 상황에서, 바꾸는 것이 유리하다는 것을 알 수 있습니다. 따라서,
   - 결론적으로, 사회자가 질문한 문(2번 문)으로 바꾸는 것이 유리합니다. 이 방식은 문을 바꾸는 것이 유리하다는 기존 몽티 홀 문제의 해답과 동일한 원리를 따릅니다.
   
이와 같이 기존 몽티 홀 문제와 비교하

In [49]:
# few-shot cot : CoT, 예시

In [50]:
system_prompt = """당신은 퍼즐 전문가입니다. 아래 문제들은 유명한 문제의 **변형**입니다.
원래 문제의 답을 그대로 적용하지 마세요. 이 문제의 조건을 정확히 읽고 답하세요.

예시:
문제 : 셔츠 1장을 말리는데 4시간이 걸립니다. 셔츠 5장을 말리면 몇 시간?
[흔한 실수] 4 x 5 = 20시간
[올바른 생각] 동시에 널면 됩니다. 답 : 4시간

이처럼 문제의 조건을 주의 깊게 읽고 단계별로 생각하세요
"""

In [51]:
"친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?"

'친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?'

In [52]:
prompt = """
당신은 물리 문제를 푸는 전문가입니다. 절대 직감이나 추측으로 답하지 말고, 정의에 따라 단계적으로 계산하세요.

문제 :
한 사람이 친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?

다음 순서를 반드시 따르세요

1. 편도 거리를 D마일이라고 두세요
2. 갈 때 걸린 시간을 계산하세요
3. 왕복 전체 평균 속도가 6mph가 되기 위한 총 시간을 계산하세요
4. 이미 사용한 시간과 필요한 총 시간을 비교하세요
5. 남은 시간으로 돌아오는 것이 가능한지 판단하세요
6. 가능하다면 필요한 속도를 계산하고, 불가능하면 왜 불가능한지 논리적으로 설명하세요
"""

In [53]:
print(llm.invoke(prompt).content)

문제를 단계적으로 해결하겠습니다.

### 1. 편도 거리를 D마일이라고 두세요
편도 거리 \( D \) 마일로 설정합니다.

### 2. 갈 때 걸린 시간을 계산하세요
걸어가는 평균 속도가 시속 3마일이므로, 편도 거리 \( D \) 마일을 걸어가는 데 걸린 시간은 다음과 같습니다:

\[
\text{시간 (t)} = \frac{D}{3} \text{ 시간}
\]

### 3. 왕복 전체 평균 속도를 6mph로 만들기 위한 총 시간을 계산하세요
왕복 거리는 \( 2D \) 마일입니다. 평균 속도가 6mph일 때, 왕복에 걸리는 총 시간은 다음과 같습니다:

\[
\text{총 시간 (T)} = \frac{2D}{6} = \frac{D}{3} \text{ 시간}
\]

### 4. 이미 사용한 시간과 필요한 총 시간을 비교하세요
갈 때 걸린 시간은 \( \frac{D}{3} \) 시간이고, 왕복 전체 평균 속도를 6mph로 유지하기 위한 총 시간도 \( \frac{D}{3} \) 시간입니다. 즉, 두 시간은 같습니다.

### 5. 남은 시간으로 돌아오는 것이 가능한지 판단하세요
왕복 전체 평균 속도를 6mph로 만들기 위해서는 돌아오는 시간을 포함해서 총 시간도 \( \frac{D}{3} \)시간에 머물러야 합니다. 

하지만, 이미 갈 때 \( \frac{D}{3} \) 시간을 사용했기 때문에 돌아올 시간은 남아 있지 않습니다. 즉, 남은 시간은 0시간입니다.

### 6. 가능하다면 필요한 속도를 계산하고, 불가능하면 왜 불가능한지 논리적으로 설명하세요
돌아오는 데 걸리는 시간을 설정해야 하지만, 이미 갈 때 다 썼기 때문에, 돌아가는 데 걸릴 수 있는 것은 없습니다. 즉, 돌아오려면 시속 무한대의 속도로 가야 하는데 이는 물리적으로 불가능합니다.

결론적으로, 친구 집까지 왕복 전체 평균 속도를 시속 6마일로 만들기 위해서는 다시 돌아오는 것이 불가능합니다.
